# 🎯 NB · Tarea de la Sesión 5 · Mi tesis frente a un modelo simple y a un modelo "tonto"
**Research Methods & Scientific Integrity in AI · UNMSM · Dra. Loveleen Gaur** · Estudiante: Félix Richar Mendoza Pumacahua

Este cuaderno resuelve las dos partes de la tarea del NB3:
1. **Reemplazar el MLP por el modelo de mi tesis** (pipeline de dos agentes: generador + verificador) y compararlo con un modelo simple y uno "tonto".
2. **Informe de 3 líneas** sobre un artículo publicado (Alenezi et al., 2026), con la verificación estadística hecha en vivo.

> ⚠️ **Parte 1 usa datos SIMULADOS.** Las tasas de detección de cada sistema son supuestos inventados para demostrar el método. **No son resultados de mi investigación.** Al final hay una celda para cargar mis resultados reales (CSV) y reutilizar el mismo análisis.

## 0 · Preparación

In [1]:
import platform, sys, numpy as np, pandas as pd, scipy
from itertools import product
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__)
SEED = 2026
N_TASKS, N_RUNS = 200, 10     # 200 tareas de verificación, cada sistema se ejecuta 10 veces

Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | scipy 1.16.3


## Parte 1 · Los cuatro sistemas
| Sistema | Papel | Qué hace |
|---|---|---|
| **Tonto** | piso | Siempre dice "cumple" (clase mayoritaria) |
| **Simple** | línea base de reglas | Verificador solo con reglas deterministas (sin LLM) |
| **Un agente** | línea base de P1 | Un solo agente genera y se autoverifica |
| **Dos agentes** | mi propuesta | Generador + verificador independiente, parcialmente determinista |

Tarea: detectar si un artefacto **viola** la conformidad (1) o **cumple** (0). Métricas de la tesis: precisión y recall (P1).

In [2]:
# SUPUESTOS INVENTADOS (solo demostración): (recall, tasa de falsos positivos)
ASSUMED = {"Dumb (siempre 'cumple')": (0.00, 0.00),
           "Simple (reglas deterministas)": (0.50, 0.05),
           "Un agente": (0.70, 0.15),
           "Dos agentes (mi propuesta)": (0.85, 0.08)}
DETERMINISTIC = {"Dumb (siempre 'cumple')", "Simple (reglas deterministas)"}   # misma salida en cada ejecución

rng0 = np.random.default_rng(SEED)
y_true = (rng0.random(N_TASKS) < 0.30).astype(int)      # 30 % de artefactos con violación
print("Violaciones reales:", y_true.sum(), "de", N_TASKS)

Violaciones reales: 53 de 200


In [3]:
def simulate():
    rows = []
    for k, (name, (rec, fpr)) in enumerate(ASSUMED.items()):
        for run in range(N_RUNS):
            # los sistemas deterministas usan la MISMA semilla en cada ejecución; los de LLM, una distinta
            seed = [SEED, 100 + k, 0 if name in DETERMINISTIC else 1 + run]   # semilla distinta de la que generó y_true
            r = np.random.default_rng(seed)
            u = r.random(N_TASKS)
            pred = np.where(y_true == 1, u < rec, u < fpr).astype(int)
            rows += [(i, run, name, y_true[i], pred[i]) for i in range(N_TASKS)]
    return pd.DataFrame(rows, columns=["task_id", "run", "system", "y_true", "y_pred"])
df = simulate()
df.head()

,task_id,run,system,y_true,y_pred
0,0,0,Dumb (siempre 'cumple'),1,0
1,1,0,Dumb (siempre 'cumple'),0,0
2,2,0,Dumb (siempre 'cumple'),0,0
3,3,0,Dumb (siempre 'cumple'),0,0
4,4,0,Dumb (siempre 'cumple'),0,0


## Evaluación honesta: una métrica por ejecución, luego **media ± DE**
Misma regla del NB1 y NB4: no se reporta una ejecución con suerte.

In [4]:
def prf(g):
    tp = ((g.y_true==1)&(g.y_pred==1)).sum(); fp = ((g.y_true==0)&(g.y_pred==1)).sum(); fn = ((g.y_true==1)&(g.y_pred==0)).sum()
    p = tp/(tp+fp) if tp+fp else np.nan; r = tp/(tp+fn) if tp+fn else np.nan
    f = 2*p*r/(p+r) if (p==p and r==r and p+r>0) else np.nan
    return pd.Series({"precision":p, "recall":r, "f1":f, "escape_rate": fn/(tp+fn)})   # escape = violaciones no detectadas

per_run = df.groupby(["system","run"]).apply(prf).reset_index()
fmt = lambda s: f"{s.mean():.3f} ± {s.std(ddof=1):.3f}" if s.notna().all() else "n/d (sin predicciones positivas)"
report = per_run.groupby("system")[["precision","recall","f1","escape_rate"]].agg(fmt)
report = report.loc[list(ASSUMED)]
report

/tmp/ipykernel_4408/923216047.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_run = df.groupby(["system","run"]).apply(prf).reset_index()


,precision,recall,f1,escape_rate
system,,,,
Dumb (siempre 'cumple'),n/d (sin predicciones positivas),0.000 ± 0.000,n/d (sin predicciones positivas),1.000 ± 0.000
Simple (reglas deterministas),0.844 ± 0.000,0.509 ± 0.000,0.635 ± 0.000,0.491 ± 0.000
Un agente,0.617 ± 0.046,0.708 ± 0.068,0.658 ± 0.048,0.292 ± 0.068
Dos agentes (mi propuesta),0.825 ± 0.092,0.858 ± 0.043,0.839 ± 0.059,0.142 ± 0.043


In [5]:
# Comparación pareada por ejecución: dos agentes vs un agente (P1 de mi protocolo)
a = per_run[per_run.system=="Dos agentes (mi propuesta)"].set_index("run")
b = per_run[per_run.system=="Un agente"].set_index("run")
rng = np.random.default_rng(SEED)
for m in ["precision","recall","f1"]:
    d = (a[m]-b[m]).values
    boot = [rng.choice(d, len(d)).mean() for _ in range(5000)]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"Δ{m:<9}: {d.mean():+.3f}  IC95% bootstrap [{lo:+.3f}, {hi:+.3f}]")
print("\nOjo: aquí 'ejecuciones' repiten las MISMAS 200 tareas; el IC solo mide el azar del LLM, no la variación entre tareas.")

Δprecision: +0.208  IC95% bootstrap [+0.133, +0.280]
Δrecall   : +0.151  IC95% bootstrap [+0.092, +0.209]
Δf1       : +0.181  IC95% bootstrap [+0.127, +0.232]

Ojo: aquí 'ejecuciones' repiten las MISMAS 200 tareas; el IC solo mide el azar del LLM, no la variación entre tareas.


### Lista de fuga y validez (la del NB3, aplicada a mi tesis)
| Pregunta del NB3 | En mi pipeline |
|---|---|
| ¿Semilla con suerte? | Repetir cada tarea **≥ 10 veces** y reportar media ± DE |
| ¿Mismo caso en entrenamiento y prueba? | Las normas y ejemplos de evaluación **no** deben estar en el prompt, ni en el contexto del generador ni del verificador |
| ¿Algún paso vio la prueba? | Fijar prompts y umbrales **antes** de ver el conjunto de evaluación; sin ajustes sobre él |
| ¿Modelo simple y tonto? | Reglas deterministas + clase mayoritaria (arriba) |
| Propio de mi caso | Fallas **correlacionadas** entre generador y verificador (prueba de estrés opcional) |
| Reproducibilidad | Registrar modelo y versión, prompts, temperatura, fecha y commit del repo |

### Mis resultados reales (cuando existan)
Sube un CSV con columnas `task_id, run, system, y_true, y_pred` y ejecuta la celda; reutiliza `prf` y `fmt` de arriba.

In [6]:
REAL_CSV = None   # p. ej. "mis_resultados.csv"  (subir desde el panel de archivos de Colab)
if REAL_CSV:
    real = pd.read_csv(REAL_CSV)
    pr = real.groupby(["system","run"]).apply(prf).reset_index()
    display(pr.groupby("system")[["precision","recall","f1","escape_rate"]].agg(fmt))
else:
    print("Sin datos reales todavía: arriba solo hay una SIMULACIÓN.")

Sin datos reales todavía: arriba solo hay una SIMULACIÓN.


## Parte 2 · Informe de 3 líneas de un artículo publicado
**Artículo:** Alenezi, Akour & Al-Omari (2026), *Agentic artificial intelligence in software engineering*, Int. J. Technology 17(4). DOI: 10.14716/ijtech.v17i4.8380 (auditado en la Sesión 6).

### Verificación en vivo: ¿puede salir p = 0.005–0.02 con n = 4 equipos?
Con n = 4 pares, el test de rangos con signo de Wilcoxon solo tiene 2⁴ = 16 combinaciones de signos. Las enumero.

In [7]:
from scipy.stats import wilcoxon
n = 4
ranks = np.arange(1, n+1)
# estadístico W+ para cada una de las 2^n asignaciones de signo
sums = np.array([sum(r for r, s in zip(ranks, signs) if s) for signs in product([0,1], repeat=n)])
p_min_two_sided = 2 * (sums >= sums.max()).sum() / len(sums)
print("Combinaciones de signos:", len(sums), "| p mínimo exacto bilateral =", p_min_two_sided)
# comprobación con SciPy: el caso más extremo posible (4 de 4 equipos mejoran)
res = wilcoxon([5, 6, 7, 8], [0, 0, 0, 0], method="exact")
print("SciPy, mejor caso con n = 4:", round(res.pvalue, 3))
print("¿p = 0.005–0.02 es posible?", p_min_two_sided <= 0.02)

Combinaciones de signos: 16 | p mínimo exacto bilateral = 0.125
SciPy, mejor caso con n = 4: 0.125
¿p = 0.005–0.02 es posible? False


In [8]:
print("INFORME DE 3 LÍNEAS · Alenezi et al. (2026)")
print("-"*72)
print("¿Mismo número?  NO verificable: sin código ni datos; solo medianas agregadas.")
print("¿Válido?        NO demostrado: p = 0.005–0.02 es imposible con n = 4 (mínimo 0.125);")
print("                el resumen y la Tabla 4 no coinciden.")
print("¿Problemas?     Censura de defectos a 30 días, ventanas de 8 y 12 semanas, sin grupo")
print("                de control, plataforma de agentes sin identificar. Puntaje 3.5/9.")
print("-"*72)
print("Conclusión: no reproducible (no se puede intentar) y no válido (los datos no lo sustentan).")

INFORME DE 3 LÍNEAS · Alenezi et al. (2026)
------------------------------------------------------------------------
¿Mismo número?  NO verificable: sin código ni datos; solo medianas agregadas.
¿Válido?        NO demostrado: p = 0.005–0.02 es imposible con n = 4 (mínimo 0.125);
                el resumen y la Tabla 4 no coinciden.
¿Problemas?     Censura de defectos a 30 días, ventanas de 8 y 12 semanas, sin grupo
                de control, plataforma de agentes sin identificar. Puntaje 3.5/9.
------------------------------------------------------------------------
Conclusión: no reproducible (no se puede intentar) y no válido (los datos no lo sustentan).


## Conclusión para mi tesis
- Mi propuesta debe compararse **siempre** con un modelo tonto, uno simple de reglas y la línea base de un agente.
- Se reportan **media ± DE** de varias ejecuciones, no una corrida.
- Publicar **código, datos, prompts y versiones**: lo que le falta al artículo auditado.

**Lectura:** Kapoor & Narayanan (2023), *Patterns* · doi.org/10.1016/j.patter.2023.100804